In [1]:
import os
import xml.etree.ElementTree as ET
from collections import OrderedDict

# ── File Paths ──────────────────────────────────────────────
FYFE_XML = "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml"
BUTCHER_XML = "/Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.butcher1911.xml"
OUTPUT_XML = "/Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.butcher1911-aligned.xml"

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

def extract_raw_text(elem):
    """Recursively grabs text while strips child element formatting tags."""
    parts = []
    if elem.text: parts.append(elem.text)
    for child in elem:
        parts.append(extract_raw_text(child))
        if child.tail: parts.append(child.tail)
    return ' '.join(''.join(parts).split())

def parse_butcher_flat(path):
    """Extracts Butcher text mapped by Chapter and Section string arrays."""
    tree = ET.parse(path)
    root = tree.getroot()
    
    # Handle flat tag structure without strict namespace constraints
    body = root.find('.//body')
    data = OrderedDict()
    
    for ch_div in body.findall('.//div[@type="textpart"]'):
        ch_n = ch_div.get('n')
        if not ch_n: continue
        data[ch_n] = OrderedDict()
        
        # In Butcher, the paragraph containers are grouped in 'section' divs or inline <p>
        sections = ch_div.findall('.//div[@subtype="section"]')
        if not sections:
            sections = ch_div.findall('.//p')
            
        for idx, sec in enumerate(sections):
            sec_n = sec.get('n') or str(idx + 1)
            data[ch_n][sec_n] = extract_raw_text(sec)
            
    return data

print("Ingesting source data frameworks...")
butcher_data = parse_butcher_flat(BUTCHER_XML)

# Parse Fyfe skeleton framework template
ET.register_namespace('', "http://www.tei-c.org/ns/1.0")
fyfe_tree = ET.parse(FYFE_XML)
fyfe_root = fyfe_tree.getroot()

# Update TEI Header metadata references to reflect Butcher's translation attributes
title_stmt = fyfe_root.find('.//tei:titleStmt', NS)
if title_stmt is not None:
    editor = title_stmt.find('tei:editor', NS)
    if editor is not None:
        editor.text = "S.H. Butcher"
        editor.set('role', 'translator')

# ── Core Alignment Transformation Engine ─────────────────────
print("Mapping Butcher content onto Fyfe TEI XML framework...")
body = fyfe_root.find('.//tei:body', NS)
text_container = body.find('tei:div', NS)

# Loop through Fyfe's structural skeleton chapters and subchapters
for ch_div in text_container.findall('tei:div[@subtype="chapter"]', NS):
    ch_n = ch_div.get('n')
    
    for subch_div in ch_div.findall('tei:div[@subtype="subchapter"]', NS):
        subch_n = subch_div.get('n')
        
        # Retrieve the matching translated passage from Butcher's data dictionary
        # Safely fall back if the section maps differ slightly across editions
        text_match = butcher_data.get(ch_n, {}).get(subch_n, "")
        
        if not text_match and subch_n == "1" and butcher_data.get(ch_n):
            # Fallback alignment logic for single long chapter segments
            text_match = " ".join(butcher_data[ch_n].values())
            
        # Clean out Fyfe's original english string values
        for p in subch_div.findall('tei:p', NS):
            subch_div.remove(p)
            
        # Clear inner child text blocks
        subch_div.text = ""
        
        # Inject the new aligned structural paragraph block element
        new_p = ET.Element('{http://www.tei-c.org/ns/1.0}p')
        new_p.text = text_match if text_match.strip() else "[Content segment missing or omitted in Butcher's 1911 edition]"
        subch_div.append(new_p)

# Save the structured XML file
fyfe_tree.write(OUTPUT_XML, encoding="UTF-8", xml_declaration=True)
print(f"[SUCCESS] Aligned file created at: {OUTPUT_XML}")

Ingesting source data frameworks...
Mapping Butcher content onto Fyfe TEI XML framework...
[SUCCESS] Aligned file created at: /Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.butcher1911-aligned.xml
